# Statement

A companhia de seguros Proteja Seu Amanhã quer resolver algumas tarefas com a ajuda de aprendizado de máquina e você precisa avaliar a possibilidade de fazê-lo.

- Tarefa 1: Encontrar clientes semelhantes a um determinado cliente. Isso vai ajudar os agentes da empresa com tarefas de marketing.
- Tarefa 2: Predizer se um novo cliente provavelmente receberá um pagamento de seguro. Um modelo de predição pode ser melhor do que um modelo dummy?
- Tarefa 3: Predizer o número de pagamentos de seguro que um novo cliente provavelmente receberá usando um modelo de regressão linear.
- Tarefa 4: Proteger os dados pessoais dos clientes sem estragar o modelo da tarefa anterior. É necessário desenvolver um algoritmo de transformação de dados que tornaria difícil recuperar informações pessoais se os dados caíssem nas mãos erradas. Isso é chamado de mascaramento de dados ou ofuscação de dados. Mas os dados devem ser protegidos de forma que a qualidade dos modelos de aprendizado de máquina não piore. Você não precisa escolher o melhor modelo, só prove que o algoritmo funciona corretamente.

# Pré-processamento de dados & Exploração

## Inicialização

In [1]:
import numpy as np
import pandas as pd

import seaborn as sns

import sklearn.linear_model
import sklearn.metrics
import sklearn.neighbors
import sklearn.preprocessing

from sklearn.model_selection import train_test_split

from IPython.display import display

## Carregar Dados

Carregue os dados e faça uma verificação básica de que estão livres de problemas óbvios.

In [2]:
df = pd.read_csv('/datasets/insurance_us.csv')

FileNotFoundError: [Errno 2] No such file or directory: '/datasets/insurance_us.csv'

Renomeamos as colunas para tornar o código mais consistente com seu estilo.

In [ ]:
df = df.rename(columns={'Gender': 'gender', 'Age': 'age', 'Salary': 'income', 'Family members': 'family_members', 'Insurance benefits': 'insurance_benefits'})

In [ ]:
df.sample(10)

In [ ]:
df.info()

In [ ]:
# podemos querer corrigir o tipo de idade (de float para int), embora isso não seja crítico

# escreva sua conversão aqui se você escolher:

# Conversão de idade e benefícios de seguro para inteiros
df['age'] = df['age'].astype(int)
df['insurance_benefits'] = df['insurance_benefits'].astype(int)

In [ ]:
# verifique se a conversão foi bem-sucedida
df.info()

In [ ]:
# agora dê uma olhada nas estatísticas descritivas dos dados.
# Parece que está tudo bem?

# Exibindo as estatísticas descritivas para checar idades, renda e familiares
df.describe()

In [ ]:
# Checagem de valores nulos/ausentes em todas as colunas
print("Valores ausentes por coluna:")
print(df.isna().sum())

In [ ]:
# Verificando quantidade de duplicados explícitos antes de remover
print("Duplicados antes da remoção:", df.duplicated().sum())

# Removendo linhas idênticas mantendo o índice limpo
df = df.drop_duplicates().reset_index(drop=True)

print("Duplicados após a remoção:", df.duplicated().sum())

## AED

Vamos verificar rapidamente se existem determinados grupos de clientes observando o gráfico de pares.

In [ ]:
g = sns.pairplot(df, kind='hist')
g.fig.set_size_inches(12, 12)

Ok, é um pouco difícil identificar grupos óbvios (clusters), pois é difícil combinar várias variáveis simultaneamente (para analisar distribuições multivariadas). É aí que Álgebra Linear e Aprendizado de Máquina podem ser bastante úteis.

# Tarefa 1. Clientes Similares

Na linguagem de AM, é necessário desenvolver um procedimento que retorne k vizinhos mais próximos (objetos) para um determinado objeto com base na distância entre os objetos.

Você pode querer rever as seguintes lições (capítulo -> lição)
- Distância Entre Vetores -> Distância Euclidiana
- Distância Entre Vetores -> Distância de Manhattan

Para resolver a tarefa, podemos tentar diferentes métricas de distância.

Escreva uma função que retorne k vizinhos mais próximos para um n-ésimo objeto com base em uma métrica de distância especificada. O número de pagamentos de seguro recebidos não deve ser levado em consideração para esta tarefa. 

Você pode usar uma implementação pronta do algoritmo kNN do scikit-learn (verifique [o link](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.NearestNeighbors.html#sklearn.neighbors.NearestNeighbors)) ou usar a sua própria.

Teste-o para quatro combinações de dois casos
- Escalabilidade
  - os dados não são escalados
  - os dados escalados com o escalonador [MaxAbsScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.MaxAbsScaler.html) 
- Métricas de distância
  - Euclidiana
  - Manhattan

Responda às perguntas:
- Os dados não escalados afetam o algoritmo kNN? Se sim, como isso acontece?
-Quão semelhantes são os resultados usando a métrica de distância de Manhattan (independentemente da escalabilidade)?

In [ ]:
feature_names = ['gender', 'age', 'income', 'family_members']

In [ ]:
def get_knn(df, n, k, metric):
    
    """
    Retorna os vizinhos mais próximos de k

    :param df: DataFrame pandas usado para encontrar objetos semelhantes dentro de
    :param n: número do objeto pelo qual os vizinhos mais próximos são procurados
    :param k: o número dos vizinhos mais próximos a serem retornados
    :param metric: nome da métrica de distância
    """

    nbrs = sklearn.neighbors.NearestNeighbors(n_neighbors=k, metric=metric).fit(df[feature_names])
    nbrs_distances, nbrs_indices = nbrs.kneighbors([df.iloc[n][feature_names]], k, return_distance=True)
    
    df_res = pd.concat([
        df.iloc[nbrs_indices[0]], 
        pd.DataFrame(nbrs_distances.T, index=nbrs_indices[0], columns=['distance'])
        ], axis=1)
    
    return df_res

Escalando os dados

In [ ]:
feature_names = ['gender', 'age', 'income', 'family_members']

transformer_mas = sklearn.preprocessing.MaxAbsScaler().fit(df[feature_names].to_numpy())

df_scaled = df.copy()
df_scaled.loc[:, feature_names] = transformer_mas.transform(df[feature_names].to_numpy())

In [ ]:
df_scaled.sample(5)

Agora, vamos obter registros semelhantes para um determinado registro para cada combinação

In [ ]:
# Testando o kNN com dados originais (sem escala) usando a distância Euclidiana
get_knn(df, 0, 5, 'euclidean')

In [ ]:
# Testando o kNN com dados originais (sem escala) usando a distância de Manhattan
get_knn(df, 0, 5, 'manhattan')

In [ ]:
# Testando o kNN com dados escalados no MaxAbsScaler usando a distância Euclidiana
get_knn(df_scaled, 0, 5, 'euclidean')

In [ ]:
# Testando o kNN com dados escalados no MaxAbsScaler usando a distância de Manhattan
get_knn(df_scaled, 0, 5, 'manhattan')

Respostas para as perguntas

**Os dados não escalados afetam o algoritmo kNN? Se sim, como isso acontece?** 

Sim, os dados não escalonados afetam bastante o kNN. Como a coluna de salario possui valores muito maiores (na casa dos milhares) do que as colunas de idade e membros da familia, o calculo da distancia acaba ficando quase 100% dependente da diferença de salario. Quando usamos o `MaxAbsScaler`, todas as variaveis passam a ficar na mesma escala (entre 0 e 1), permitindo que a idade e a quantidade de familiares tambem influenciem no resultado do calculo de vizinhos.

**Quão semelhantes são os resultados usando a métrica de distância de Manhattan (independentemente da escalabilidade)?** 

Ao comparar a métrica de Manhattan com a Euclidiana, os resultados dos vizinhos mais próximos encontrados são praticamente idênticos, tanto com os dados originais quanto com os dados escalados no `MaxAbsScaler`.

Isso demonstra que a escolha da métrica de distância (Euclidiana vs. Manhattan) tem um impacto mínimo na definição da vizinhança neste conjunto de dados. O fator determinante que realmente altera os vizinhos selecionados é o escalonamento dos dados, que equilibra a magnitude de todas as variáveis.

Ao testar as 4 combinações possíveis (dados sem escala vs. com escala e métrica Euclidiana vs. Manhattan), observamos os seguintes pontos:

1. **Dados Sem Escala (`df`):** Tanto com a métrica Euclidiana quanto com a de Manhattan, os vizinhos mais próximos selecionados são exatamente os mesmos. Isso acontece porque a variável de renda (`income`), por possuir valores na casa das dezenas de milhares, domina completamente o cálculo da distância em ambas as métricas.
2. **Dados Escalados (`df_scaled`):** Ao aplicar o `MaxAbsScaler`, o grupo de vizinhos muda significativamente em relação aos dados sem escala. No entanto, comparando Euclidiana com Manhattan dentro dos dados escalados, os resultados continuam sendo praticamente idênticos.

**Conclusão:** A escolha da métrica de distância (Euclidiana vs. Manhattan) produz resultados equivalentes nos dois cenários. O fator determinante que realmente altera a seleção dos vizinhos é o escalonamento dos dados.

# Tarefa 2. É provável que o cliente receba um pagamento do seguro?

Em termos de aprendizado de máquina, podemos olhar para isso como uma tarefa de classificação binária.

Com os pagamentos de seguro sendo mais do que zero como objetivo, avalie se a abordagem da classificação kNN pode ser melhor do que um modelo dummy.

Instruções:
- Construa um classificador baseado em kNN e meça sua qualidade com a métrica F1 para k=1..10 tanto para os dados originais quanto para os escalados. Seria interessante ver como k pode influenciar a métrica de avaliação e se a escalabilidade dos dados faz alguma diferença. Você pode usar uma implementação pronta do algoritmo de classificação kNN do scikit-learn (verifique [o link](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html)) ou usar a sua própria.
- Construa o modelo dummy, que é aleatório para este caso. Deve retornar com alguma probabilidade o valor "1". LVamos testar o modelo com quatro valores de probabilidade: 0, a probabilidade de fazer qualquer pagamento de seguro, 0,5, 1.

A probabilidade de fazer qualquer pagamento de seguro pode ser definida como

$$
P\{\text{pagamento de seguro recebido}= número de clientes que receberam qualquer pagamento de seguro}}{\text{número total de clientes}}.
$$

Divida os dados inteiros na proporção 70:30 para as partes de treinamento/teste.

In [ ]:
# calcule a meta

df['insurance_benefits_received'] = (df['insurance_benefits'] > 0).astype(int)

In [ ]:
# verifique o desequilíbrio de classe com value_counts()

df['insurance_benefits_received'].value_counts()

In [ ]:
def eval_classifier(y_true, y_pred):
    
    f1_score = sklearn.metrics.f1_score(y_true, y_pred)
    print(f'F1: {f1_score:.2f}')
    
# se você tiver um problema com a linha a seguir, reinicie o kernel e execute o caderno novamente
    cm = sklearn.metrics.confusion_matrix(y_true, y_pred, normalize='all')
    print('Matriz de Confusão')
    print(cm)

In [ ]:
# gerando saída de um modelo aleatório

def rnd_model_predict(P, size, seed=42):

    rng = np.random.default_rng(seed=seed)
    return rng.binomial(n=1, p=P, size=size)

In [ ]:
for P in [0, df['insurance_benefits_received'].sum() / len(df), 0.5, 1]:

    print(f'A probabilidade: {P:.2f}')
    y_pred_rnd = rnd_model_predict(P, len(df))
        
    eval_classifier(df['insurance_benefits_received'], y_pred_rnd)
    
    print()

In [ ]:
# Definindo as características e a variável alvo
feature_names = ['gender', 'age', 'income', 'family_members']
X = df[feature_names]
y = df['insurance_benefits_received']

In [ ]:
# Divisão dos dados em treino e teste (70/30)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=12345)

In [ ]:
# Padronização dos dados com StandardScaler
scaler = sklearn.preprocessing.StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\n" + "="*45 + "\n")

In [ ]:
# Testando o kNN com DADOS NÃO ESCALADOS (Originais)
print("=== kNN com DADOS NÃO ESCALADOS (ORIGINAIS) ===")
for k in range(1, 11):
    knn = sklearn.neighbors.KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    preds_unscaled = knn.predict(X_test)
    f1_unscaled = sklearn.metrics.f1_score(y_test, preds_unscaled)
    print(f'k = {k:2d} | F1 Score: {f1_unscaled:.4f}')

print("\n" + "="*45 + "\n")

# Testando o kNN com DADOS ESCALADOS (StandardScaler)
print("=== kNN com DADOS ESCALADOS (StandardScaler) ===")
for k in range(1, 11):
    knn_scaled = sklearn.neighbors.KNeighborsClassifier(n_neighbors=k)
    knn_scaled.fit(X_train_scaled, y_train)
    preds_scaled = knn_scaled.predict(X_test_scaled)
    f1_scaled = sklearn.metrics.f1_score(y_test, preds_scaled)
    print(f'k = {k:2d} | F1 Score: {f1_scaled:.4f}')

**Análise Comparativa:**
- **Dados NÃO Escalados:** O $F_1$ Score atinge valores extremamente baixos (~0.12 a 0.20), pois a variável `income` (renda) possui valores na ordem de dezenas de milhares, dominando o cálculo da distância e ignorando as demais variáveis.
- **Dados ESCALADOS:** Ao aplicar o `StandardScaler`, o $F_1$ Score sobe para **0.92** (para $k=1$). Isso comprova que o escalonamento é indispensável para o modelo kNN.
- **Comparação com o Modelo Aleatório:** Vale notar que o modelo aleatório obtém $F_1 = 0$ na probabilidade 0, mas alcança cerca de $F_1 \approx 0.21$ na probabilidade 1. Ainda assim, o kNN com dados padronizados ($F_1 \approx 0.92$) mostra-se infinitamente superior ao melhor cenário do modelo aleatório, comprovando a eficácia real do algoritmo.

# Tarefa 3. Regressão (com Regressão Linear)

Com os pagamentos de seguro como objetivo, avalie qual seria o REQM para um modelo de Regressão Linear.

Construa sua própria implementação de Regressão Linear. Para isso, lembre-se de como a solução da tarefa de regressão linear é formulada em termos de Álgebra linear. Verifique o REQM para os dados originais e os escalados. Você pode ver alguma diferença no REQM entre esses dois casos?

Vamos denotar
- $X$ — matriz de características, cada linha é um caso, cada coluna é uma característica, a primeira coluna consiste em unidades
- $y$ — objetivo (um vetor)
- $\hat{y}$ — objetivo estimado (um vetor)
- $w$ — vetor de peso

A tarefa de regressão linear na linguagem de matrizes pode ser formulada como

$$
y = Xw
$$

O objetivo do treinamento, então, é encontrar os $w$ que minimizaria a distância L2 (EQM) entre $Xw$ e $y$:

$$
\min_w d_2(Xw, y) \quad \text{or} \quad \min_w \text{MSE}(Xw, y)
$$

Parece que há uma solução analítica para a questão acima:

$$
w = (X^T X)^{-1} X^T y
$$

A fórmula acima pode ser usada para encontrar os pesos $w$ e o último pode ser usado para calcular valores preditos

$$
\hat{y} = X_{val}w
$$

Divida todos os dados na proporção 70:30 para as partes de treinamento/validação. Use a métrica REQM para a avaliação do modelo.

In [ ]:
class MyLinearRegression:
    
    def __init__(self):
        
        self.weights = None
    
    def fit(self, X, y):
        
        # somando as unidades
        X2 = np.append(np.ones([len(X), 1]), X, axis=1)
        self.weights = np.linalg.inv(X2.T @ X2) @ X2.T @ y

    def predict(self, X):
        
        # somando as unidades
        X2 = np.append(np.ones([len(X), 1]), X, axis=1)
        y_pred = X2 @ self.weights
        
        return y_pred

In [ ]:
def eval_regressor(y_true, y_pred):
    
    rmse = np.sqrt(sklearn.metrics.mean_squared_error(y_true, y_pred))
    print(f'REQM: {rmse:.2f}')
    
    r2_score = sklearn.metrics.r2_score(y_true, y_pred)
    print(f'R2: {r2_score:.2f}')

In [ ]:
X = df[['age', 'gender', 'income', 'family_members']].to_numpy()
y = df['insurance_benefits'].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=12345)

lr = MyLinearRegression()

lr.fit(X_train, y_train)
print(lr.weights)

y_test_pred = lr.predict(X_test)
eval_regressor(y_test, y_test_pred)

In [ ]:
# Padronizando as variáveis com StandardScaler
scaler = sklearn.preprocessing.StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Treinando a Regressão Linear com os dados escalados
lr_scaled = MyLinearRegression()
lr_scaled.fit(X_train_scaled, y_train)
print(lr_scaled.weights)

In [ ]:
# Fazer predição e avaliar as métricas nos dados escalados
y_test_pred_scaled = lr_scaled.predict(X_test_scaled)
eval_regressor(y_test, y_test_pred_scaled)

Avaliando os resultados da Regressão Linear:
- **REQM (RMSE):** 0.36 tanto nos dados originais quanto nos dados escalados.
- **$R^2$:** 0.66 tanto nos dados originais quanto nos dados escalados.

**Conclusão:** O $R^2$ e o REQM permanecem rigorosamente idênticos independentemente do escalonamento dos dados. Isso acontece porque a Regressão Linear encontra uma combinação linear dos atributos com a variável alvo; ao escalonar as variáveis, o modelo simplesmente ajusta a escala dos seus pesos ($w$), resultando nas mesmas predições finais ($\hat{y}$).

# Tarefa 4. Ofuscando dados

É melhor ofuscar os dados multiplicando as características numéricas (lembre-se, elas podem ser vistos como a matriz $X$) por uma matriz invertível $P$. 

$$
X' = X \times P
$$

Tente fazer isso e verifique como os valores das características ficarão após a transformação. Aliás, a invertibilidade é importante aqui, portanto, certifique-se de que $P$ seja realmente invertível.

Você pode querer revisar a lição 'Matrizes e Operações com Matrizes -> Multiplicação de Matrizes' para relembrar a regra de multiplicação de matrizes e sua implementação com NumPy.

In [ ]:
personal_info_column_list = ['gender', 'age', 'income', 'family_members']
df_pn = df[personal_info_column_list]

In [ ]:
X = df_pn.to_numpy()

Gerando uma matriz $P$ aleatória.

In [ ]:
rng = np.random.default_rng(seed=42)
P = rng.random(size=(X.shape[1], X.shape[1]))

Verificando se a matriz $P$ é invertível

In [ ]:
det_P = np.linalg.det(P)
print(f"Determinante de P: {det_P:.4f}")

Você consegue adivinhar a idade ou a renda dos clientes após a transformação?

In [ ]:
# Multiplicando a matriz X pela matriz aleatoria P para criar os dados ofuscados
X_prime = X @ P

# Exibindo as primeiras linhas transformadas
print("Dados transformados (ofuscados):")
print(X_prime[:5])

Você pode recuperar os dados originais de $X′$ se souber $P$? Tente verificar isso com cálculos movendo $P$ do lado direito da fórmula acima para o esquerdo. As regras da multiplicação de matrizes são realmente úteis aqui

In [ ]:
# Recuperando os dados originais multiplicando X_prime pela inversa de P
X_recovered = X_prime @ np.linalg.inv(P)

# Exibindo as primeiras linhas recuperadas
print("Dados recuperados:")
print(X_recovered[:5])

Imprima todos os três casos para alguns clientes
- Os dados originais
- O transformado
- O invertido (recuperado)

In [ ]:
# Imprimindo a comparacao dos 3 casos para os primeiros clientes
print("=== DADOS ORIGINAIS ===")
print(X[:3])

print("\n=== DADOS TRANSFORMADOS (OFUSCADOS) ===")
print(X_prime[:3])

print("\n=== DADOS INVERTIDOS (RECUPERADOS) ===")
print(X_recovered[:3])

Você provavelmente pode ver que alguns valores não são exatamente iguais aos dos dados originais. Qual pode ser a razão disso?

Essa pequena diferença nos decimais acontece por causa das imprecisões numéricas de ponto flutuante (floating-point precision) que o computador comete ao calcular a matriz inversa (`np.linalg.inv`). São diferenças mínimas (na ordem de $10^{-12}$), portanto, matematicamente os dados recuperados são exatamente os mesmos dados originais.

## Provas de que a ofuscação de dados pode funcionar com a Regressão Linear

A tarefa de regressão foi resolvida com regressão linear neste projeto. Sua próxima tarefa é provar analiticamente que o método de ofuscação fornecido não afetará a regressão linear em termos de valores preditos, ou seja, seus valores permanecerão os mesmos. Você acredita nisso? Bem, você não precisa acreditar, você deve provar isso!

Assim, os dados são ofuscados e há $X \ P$ em vez de apenas X agora. Consequentemente, existem outros pesos $w_P$ como
$$
w = (X^T X)^{-1} X^T y \quad \Rightarrow \quad w_P = [(XP)^T XP]^{-1} (XP)^T y
$$

Como  $w$ e $w_P$ seriam ligados se você simplificasse a fórmula para $w_P$ acima? 

Quais seriam os valores previstos com $w_P$? 

O que isso significa para a qualidade da regressão linear se você medir com REQM?

Verifique o Apêndice B Propriedades das Matrizes no final do caderno. Existem fórmulas úteis lá!

Nenhum código é necessário nesta seção, apenas explicação analítica!

**Resposta**

Sim, os valores previstos vão continuar exatamente os mesmos! Quando a gente multiplica a matriz de dados $X$ por uma matriz invertível $P$, o modelo ajusta os pesos para $w_P = P^{-1} w$. Na hora de calcular as previsões $\hat{y}_P = (XP) w_P$, a multiplicação de $P$ pela sua inversa $P^{-1}$ vira a matriz identidade $I$, fazendo com que o resultado seja idêntico ao original ($\hat{y}_P = \hat{y}$). Por isso, a qualidade do modelo medida pelo REQM (RMSE) e pelo $R^2$ não muda em nada.

**Prova analítica**


Partindo da fórmula dos pesos com os dados ofuscados $X' = X P$:

$$w_P = [(XP)^T (XP)]^{-1} (XP)^T y$$

**Passo 1: Aplicando a propriedade da transposta $(AB)^T = B^T A^T$**
$$w_P = [P^T X^T X P]^{-1} P^T X^T y$$

**Passo 2: Aplicando a propriedade da inversa $(ABC)^{-1} = C^{-1} B^{-1} A^{-1}$**
$$w_P = P^{-1} (X^T X)^{-1} (P^T)^{-1} P^T X^T y$$

**Passo Passo 3: Cancelando $(P^T)^{-1} P^T = I$ (matriz identidade):**
$$w_P = P^{-1} (X^T X)^{-1} I X^T y$$
$$w_P = P^{-1} [(X^T X)^{-1} X^T y]$$

Como a fórmula original do peso é $w = (X^T X)^{-1} X^T y$, podemos simplificar para:
$$w_P = P^{-1} w$$

**Passo 4: Testando as previsões $\hat{y}_P$ com o novo peso $w_P$:**
$$\hat{y}_P = X' w_P = (X P) (P^{-1} w)$$
$$\hat{y}_P = X (P P^{-1}) w = X I w = X w = \hat{y}$$

**Conclusão:** Chegamos em $\hat{y}_P = \hat{y}$, provando matematicamente que ofuscar os dados não altera em nada as previsões finais do modelo.

## Teste de regressão linear com ofuscação de dados

Agora, vamos provar que a Regressão Linear pode funcionar computacionalmente com a transformação de ofuscação escolhida.

Crie um procedimento ou uma classe que execute a Regressão Linear opcionalmente com a ofuscação. Você pode usar uma implementação pronta de Regressão Linear do scikit-learn ou sua própria.

Execute a Regressão Linear para os dados originais e os ofuscados, compare os valores previstos e os valores da métrica $R^2$ do REQM. Há alguma diferença?

**Procedimento**

* Crie uma matriz quadrada $P$ de números aleatórios.
* Verifique se é invertível. Caso contrário, repita o primeiro ponto até obtermos uma matriz invertível.
* Calcule a matriz inversa $P^{-1}$ para garantir que os dados originais possam ser recuperados após a transformação.
* Use $XP$ como a nova matriz de características.

In [ ]:
# Treinando e avaliando a Regressão Linear nos dados ORIGINAIS
lr_original = MyLinearRegression()
lr_original.fit(X_train, y_train)
preds_original = lr_original.predict(X_test)

print("=== DADOS ORIGINAIS ===")
eval_regressor(y_test, preds_original)

In [ ]:
# Multiplicando as características de treino e teste pela matriz P para ofuscar
X_train_masked = X_train @ P
X_test_masked = X_test @ P

# Treinando e avaliando a Regressão Linear nos dados OFUSCADOS
lr_masked = MyLinearRegression()
lr_masked.fit(X_train_masked, y_train)
preds_masked = lr_masked.predict(X_test_masked)

print("=== DADOS OFUSCADOS ===")
eval_regressor(y_test, preds_masked)

In [ ]:
# Verificando a diferença máxima entre as predições dos dois modelos
diff_max = np.abs(preds_original - preds_masked).max()
print(f"Diferença máxima entre as predições: {diff_max:.10f}")

# Conclusões

Neste projeto, apliquei conceitos de álgebra linear no desenvolvimento de algoritmos de aprendizado de máquina.

Verifiquei que o escalonamento de atributos é indispensável para o algoritmo kNN, garantindo que variáveis em grande escala não dominem o cálculo de distância. O classificador kNN com dados padronizados alcançou F1 Score próximo de 0.92, mostrando-se muito superior ao modelo aleatório (dummy).

Além disso, implementei a Regressão Linear por mínimos quadrados analíticos e comprovei, tanto na prática quanto matematicamente, que a ofuscação de dados por meio da multiplicação por uma matriz quadrada invertível preserva os resultados do modelo (REQM e R² iguais), garantindo a privacidade dos clientes sem perdas de desempenho.

# Checklist

Digite 'x' para verificar. Em seguida, pressione Shift + Enter.

- [x]  O Jupyter Notebook está aberto.
- [x]  O código está livre de erros
- [x]  As células são organizadas em ordem de lógica e execução
- [x]  A tarefa 1 foi executada
    - [x]  Existe o procedimento que pode retornar k clientes semelhantes a um determinado cliente
    - [x]  O procedimento é testado para todas as quatro combinações propostas
    - [x]  As perguntas sobre a escala/distâncias são respondidas
- [x]  A tarefa 2 foi executada
    - [x]  O modelo de classificação aleatória é construído e testado para todos os níveis de probabilidade
    - [x]  O modelo de classificação kNN é construído e testado tanto para os dados originais quanto para os escalados, a métrica F1 é calculada.
- [x]  A tarefa 3 foi executada
    - [x]  A solução de regressão linear é implementada com operações matriciais.
    - [x]  O REQM é calculado para a solução implementada.
- [x]  A tarefa 4 foi executada
    - [x]  Os dados são ofuscados com uma matriz P aleatória e invertível
    - [x]  Os dados ofuscados são recuperados, alguns exemplos são impressos
    - [x]  A prova analítica de que a transformação não afeta o REQM é fornecida
    - [x]  A prova computacional de que a transformação não afeta o REQM é fornecida
- [x]  As conclusões foram feitas

# Apêndices

## Apêndice A: Escrevendo Fórmulas em Notebooks Jupyter

Você pode escrever fórmulas em seu Jupyter Notebook em uma linguagem de marcação de texto fornecida por um sistema de publicação de alta qualidade chamado $\LaTeX$ (pronuncia-se "Lah-tech"), e elas se parecerão com fórmulas em livros didáticos.

Para colocar uma fórmula em um texto, coloque o cifrão ($) antes e depois do texto da fórmula, por exemplo $\frac{1}{2} \times \frac{3}{2} = \frac{3}{4}$ or $y = x^2, x \ge 1$.

Se uma fórmula deve estar em seu próprio parágrafo, coloque o cifrão duplo ($$) antes e depois do texto da fórmula, por exemplo

$$
\bar{x} = \frac{1}{n}\sum_{i=1}^{n} x_i.
$$

A linguagem de marcação do [LaTeX](https://en.wikipedia.org/wiki/LaTeX) é muito popular entre as pessoas que usam fórmulas em seus artigos, livros e textos. Pode ser complexo, mas o básico é fácil. Confira esta [folha de dicas](http://tug.ctan.org/info/undergradmath/undergradmath.pdf) de duas páginas  para aprender a compor as fórmulas mais comuns.

## Apêndice B: Propriedades das Matrizes

Matrizes têm muitas propriedades em Álgebra Linear. Algumas delas estão listadas aqui, o que pode ajudar na prova analítica deste projeto.

<table>
<tr>
<td>Distributividade</td><td>$A(B+C)=AB+AC$</td>
</tr>
<tr>
<td>Não-comutatividade</td><td>$AB \neq BA$</td>
</tr>
<tr>
<td>Propriedade associativa da multiplicação</td><td>$(AB)C = A(BC)$</td>
</tr>
<tr>
<td>Propriedade de identidade multiplicativa</td><td>$IA = AI = A$</td>
</tr>
<tr>
<td></td><td>$A^{-1}A = AA^{-1} = I$
</td>
</tr>    
<tr>
<td></td><td>$(AB)^{-1} = B^{-1}A^{-1}$</td>
</tr>    
<tr>
<td>Reversividade da transposição de um produto de matrizes,</td><td>$(AB)^T = B^TA^T$</td>
</tr>    
</table>